# Balanced K-Fold Generation for Steatosis Dataset

This notebook combines steatosis analysis (count and size) to create balanced k-fold splits.

**Pipeline:**
1. Analysis of all masks in the `manual` folder
2. Metrics calculation: steatosis count, total area, average size
3. Percentile-based categorization
4. Balanced stratified fold creation

In [1]:
# Imports
import os
import cv2
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from IPython.display import display, HTML

## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Modify these paths as needed
# =============================================================================

# Project base directory
BASE_DIR = Path("")

# Directory with original images
IMAGES_DIR = BASE_DIR / "images"

# Directory with manual masks (annotated steatosis)
MANUAL_DIR = BASE_DIR / "manual"

# Output directory for results
OUTPUT_DIR = BASE_DIR / "kfold_analysis_output"

# K-Fold Configuration
K_FOLDS = 10  # Number of folds
RANDOM_SEED = 42  # Seed for reproducibility

# Check folder existence
print("=" * 60)
print("CONFIGURATION CHECK")
print("=" * 60)
print(f"Images dir: {IMAGES_DIR}")
print(f"   Exists: {IMAGES_DIR.exists()}")
print(f"Manual dir: {MANUAL_DIR}")
print(f"   Exists: {MANUAL_DIR.exists()}")

if MANUAL_DIR.exists():
    mask_files = list(MANUAL_DIR.glob("*.png"))
    print(f"\nFound {len(mask_files)} PNG masks in manual/")

## Phase 1: Detailed Steatosis Analysis

Analyzes each mask to extract:
- Number of steatosis (connected components)
- Total white area (steatosis coverage)
- Average steatosis size
- Size distribution (min, max, median)

In [3]:
def analyze_steatosis_detailed(mask_path: Path) -> dict:
    """
    Analyzes a mask for steatosis count and size.
    
    Returns:
        Dictionary with detailed metrics
    """
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    
    if mask is None:
        raise ValueError(f"Cannot read image: {mask_path}")
    
    # Binarize
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    
    # Connected components with statistics
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary)
    
    # Subtract 1 for background
    num_steatosis = num_labels - 1
    
    # Calculate total image area
    total_pixels = mask.shape[0] * mask.shape[1]
    
    # Calculate total white area
    total_white_area = np.sum(binary > 0)
    white_percentage = (total_white_area / total_pixels) * 100
    
    # Individual steatosis sizes (exclude background at index 0)
    if num_steatosis > 0:
        steatosis_sizes = stats[1:, cv2.CC_STAT_AREA]
        avg_size = np.mean(steatosis_sizes)
        min_size = np.min(steatosis_sizes)
        max_size = np.max(steatosis_sizes)
        median_size = np.median(steatosis_sizes)
        std_size = np.std(steatosis_sizes)
    else:
        steatosis_sizes = []
        avg_size = 0
        min_size = 0
        max_size = 0
        median_size = 0
        std_size = 0
    
    return {
        "filename": mask_path.name,
        "num_steatosis": num_steatosis,
        "total_white_area": int(total_white_area),
        "white_percentage": round(white_percentage, 4),
        "avg_steatosis_size": round(avg_size, 2),
        "min_steatosis_size": int(min_size),
        "max_steatosis_size": int(max_size),
        "median_steatosis_size": round(median_size, 2),
        "std_steatosis_size": round(std_size, 2),
        "individual_sizes": [int(s) for s in steatosis_sizes]
    }

In [ ]:
# Analyze all masks
print("=" * 60)
print("PHASE 1: DETAILED STEATOSIS ANALYSIS")
print("=" * 60)

mask_files = sorted(MANUAL_DIR.glob("*.png"))
print(f"\nAnalyzing {len(mask_files)} masks...")

results = []
errors = []

for i, mask_path in enumerate(mask_files):
    try:
        data = analyze_steatosis_detailed(mask_path)
        results.append(data)
        
        if (i + 1) % 100 == 0:
            print(f"  Processed {i + 1}/{len(mask_files)} masks...")
            
    except Exception as e:
        errors.append({"file": mask_path.name, "error": str(e)})
        print(f"  Error in {mask_path.name}: {e}")

print(f"\nAnalysis completed: {len(results)} masks processed")
if errors:
    print(f"WARNING:  {len(errors)} errors found")

In [ ]:
# Create DataFrame and display statistics
df = pd.DataFrame(results)

# Remove individual_sizes column for main analysis
individual_sizes_data = {r["filename"]: r["individual_sizes"] for r in results}
df = df.drop(columns=["individual_sizes"])

print("=" * 60)
print("BASIC STATISTICS")
print("=" * 60)

# Statistics for non-black masks
non_black = df[df["num_steatosis"] > 0]

print(f"\nTotal masks: {len(df)}")
print(f"   Black masks (0 steatosis): {len(df) - len(non_black)} ({(len(df) - len(non_black))/len(df)*100:.1f}%)")
print(f"   Masks with steatosis: {len(non_black)} ({len(non_black)/len(df)*100:.1f}%)")

print(f"\nCount Statistics (non-black):")
print(f"   Min: {non_black['num_steatosis'].min()}")
print(f"   Max: {non_black['num_steatosis'].max()}")
print(f"   Mean: {non_black['num_steatosis'].mean():.2f}")
print(f"   Median: {non_black['num_steatosis'].median():.1f}")

print(f"\nWhite Area % Statistics (non-black):")
print(f"   Min: {non_black['white_percentage'].min():.4f}%")
print(f"   Max: {non_black['white_percentage'].max():.4f}%")
print(f"   Mean: {non_black['white_percentage'].mean():.4f}%")

print(f"\nAverage Size Statistics (non-black):")
print(f"   Min: {non_black['avg_steatosis_size'].min():.2f} px")
print(f"   Max: {non_black['avg_steatosis_size'].max():.2f} px")
print(f"   Mean: {non_black['avg_steatosis_size'].mean():.2f} px")

## Phase 2: Categorization

Categorizes masks based on percentiles (33rd and 66th) for:
- **Count**: black, few, medium, many
- **Area**: no_area, small_area, medium_area, large_area
- **Average size**: no_size, small_avg, medium_avg, large_avg

In [ ]:
# Calculate percentile-based thresholds
thresholds = {
    "count_p33": np.percentile(non_black["num_steatosis"], 33),
    "count_p66": np.percentile(non_black["num_steatosis"], 66),
    "area_p33": np.percentile(non_black["white_percentage"], 33),
    "area_p66": np.percentile(non_black["white_percentage"], 66),
    "avg_size_p33": np.percentile(non_black["avg_steatosis_size"], 33),
    "avg_size_p66": np.percentile(non_black["avg_steatosis_size"], 66),
}

print("=" * 60)
print("CALCULATED THRESHOLDS (33rd and 66th Percentiles)")
print("=" * 60)
print(f"\nCount:")
print(f"   few ≤ {thresholds['count_p33']:.0f}, medium ≤ {thresholds['count_p66']:.0f}, many > {thresholds['count_p66']:.0f}")
print(f"\nArea %:")
print(f"   small ≤ {thresholds['area_p33']:.4f}%, medium ≤ {thresholds['area_p66']:.4f}%, large > {thresholds['area_p66']:.4f}%")
print(f"\n Dimensione Mean:")
print(f"   small ≤ {thresholds['avg_size_p33']:.2f}px, medium ≤ {thresholds['avg_size_p66']:.2f}px, large > {thresholds['avg_size_p66']:.2f}px")

In [ ]:
# Categorization functions
def categorize_by_count(num_steatosis: int, thresholds: dict) -> str:
    """Categorize by steatosis count."""
    if num_steatosis == 0:
        return "black"
    elif num_steatosis <= thresholds["count_p33"]:
        return "few"
    elif num_steatosis <= thresholds["count_p66"]:
        return "medium"
    else:
        return "many"

def categorize_by_area(white_percentage: float, thresholds: dict) -> str:
    """Categorize by white area percentage."""
    if white_percentage == 0:
        return "no_area"
    elif white_percentage <= thresholds["area_p33"]:
        return "small_area"
    elif white_percentage <= thresholds["area_p66"]:
        return "medium_area"
    else:
        return "large_area"

def categorize_by_avg_size(avg_size: float, thresholds: dict) -> str:
    """Categorize by average steatosis size."""
    if avg_size == 0:
        return "no_size"
    elif avg_size <= thresholds["avg_size_p33"]:
        return "small_avg"
    elif avg_size <= thresholds["avg_size_p66"]:
        return "medium_avg"
    else:
        return "large_avg"

# Apply categorization
df["count_category"] = df["num_steatosis"].apply(lambda x: categorize_by_count(x, thresholds))
df["area_category"] = df["white_percentage"].apply(lambda x: categorize_by_area(x, thresholds))
df["avg_size_category"] = df["avg_steatosis_size"].apply(lambda x: categorize_by_avg_size(x, thresholds))

# Combined category for stratification
df["combined_category"] = df["count_category"] + "_" + df["area_category"]

# Normalized score for additional flexibility
df["normalized_count"] = df["num_steatosis"] / df["num_steatosis"].max() if df["num_steatosis"].max() > 0 else 0
df["normalized_area"] = df["white_percentage"] / df["white_percentage"].max() if df["white_percentage"].max() > 0 else 0
df["combined_score"] = 0.5 * df["normalized_count"] + 0.5 * df["normalized_area"]

print("Categorization completed!")

In [ ]:
# Show distribution by category
print("=" * 60)
print("DISTRIBUTION BY CATEGORY")
print("=" * 60)

print("\nBy Count:")
count_dist = df["count_category"].value_counts()
for cat in ["black", "few", "medium", "many"]:
    if cat in count_dist:
        print(f"   {cat:12s}: {count_dist[cat]:4d} ({count_dist[cat]/len(df)*100:5.1f}%)")

print("\nBy Area:")
area_dist = df["area_category"].value_counts()
for cat in ["no_area", "small_area", "medium_area", "large_area"]:
    if cat in area_dist:
        print(f"   {cat:12s}: {area_dist[cat]:4d} ({area_dist[cat]/len(df)*100:5.1f}%)")

print("\nCombined Categories (Count + Area):")
combined_dist = df["combined_category"].value_counts().sort_index()
for cat, count in combined_dist.items():
    print(f"   {cat:25s}: {count:4d} ({count/len(df)*100:5.1f}%)")

## Phase 3: Visualizations

In [ ]:
# Create output directory
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Figure 1: Distribution histograms
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Count distribution
ax = axes[0, 0]
ax.hist(df["num_steatosis"], bins=50, edgecolor='black', alpha=0.7)
ax.axvline(thresholds["count_p33"], color='r', linestyle='--', label=f'P33={thresholds["count_p33"]:.0f}')
ax.axvline(thresholds["count_p66"], color='g', linestyle='--', label=f'P66={thresholds["count_p66"]:.0f}')
ax.set_xlabel('Number of Steatosis')
ax.set_ylabel('Frequency')
ax.set_title('Steatosis Count Distribution')
ax.legend()

# Area distribution
ax = axes[0, 1]
ax.hist(non_black["white_percentage"], bins=50, edgecolor='black', alpha=0.7, color='orange')
ax.axvline(thresholds["area_p33"], color='r', linestyle='--', label=f'P33={thresholds["area_p33"]:.4f}%')
ax.axvline(thresholds["area_p66"], color='g', linestyle='--', label=f'P66={thresholds["area_p66"]:.4f}%')
ax.set_xlabel('White Area %')
ax.set_ylabel('Frequency')
ax.set_title('Area % Distribution (non-black)')
ax.legend()

# Average size distribution
ax = axes[1, 0]
ax.hist(non_black["avg_steatosis_size"], bins=50, edgecolor='black', alpha=0.7, color='green')
ax.axvline(thresholds["avg_size_p33"], color='r', linestyle='--', label=f'P33={thresholds["avg_size_p33"]:.0f}px')
ax.axvline(thresholds["avg_size_p66"], color='g', linestyle='--', label=f'P66={thresholds["avg_size_p66"]:.0f}px')
ax.set_xlabel('Average Steatosis Size (pixels)')
ax.set_ylabel('Frequency')
ax.set_title('Average Size Distribution (non-black)')
ax.legend()

# Combined score distribution
ax = axes[1, 1]
ax.hist(df["combined_score"], bins=50, edgecolor='black', alpha=0.7, color='purple')
ax.set_xlabel('Combined Score (Count + Area)')
ax.set_ylabel('Frequency')
ax.set_title('Combined Score Distribution')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "distribution_histograms.png", dpi=150)
plt.show()
print(f"Saved: {OUTPUT_DIR / 'distribution_histograms.png'}")

In [ ]:
# Combined category heatmap
fig, ax = plt.subplots(figsize=(10, 8))

count_cats = ["black", "few", "medium", "many"]
area_cats = ["no_area", "small_area", "medium_area", "large_area"]

heatmap_data = np.zeros((len(count_cats), len(area_cats)))
for i, cc in enumerate(count_cats):
    for j, ac in enumerate(area_cats):
        heatmap_data[i, j] = len(df[(df["count_category"] == cc) & (df["area_category"] == ac)])

im = ax.imshow(heatmap_data, cmap='YlOrRd')
ax.set_xticks(range(len(area_cats)))
ax.set_xticklabels(area_cats, rotation=45, ha='right')
ax.set_yticks(range(len(count_cats)))
ax.set_yticklabels(count_cats)
ax.set_xlabel('Area Category')
ax.set_ylabel('Count Category')
ax.set_title('Combined Category Heatmap (Count x Area)')

# Annotations
for i in range(len(count_cats)):
    for j in range(len(area_cats)):
        text = ax.text(j, i, int(heatmap_data[i, j]),
                      ha="center", va="center", 
                      color="black" if heatmap_data[i, j] < 50 else "white",
                      fontweight='bold')

plt.colorbar(im, ax=ax, label='Number of Masks')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "category_heatmap.png", dpi=150)
plt.show()
print(f"Saved: {OUTPUT_DIR / 'category_heatmap.png'}")

## Phase 4: Stratified K-Fold Creation

In [ ]:
def stratified_kfold_split(df: pd.DataFrame, k: int, seed: int = 42) -> tuple:
    """
    Creates balanced stratified k-folds.
    
    Args:
        df: DataFrame con 'filename' e 'combined_category'
        k: Numero di fold
        seed: Seed random
        
    Returns:
        Tupla (folds, fold_counts)
    """
    np.random.seed(seed)
    
    # Create stratification column
    df = df.copy()
    df["strat_group"] = df["combined_category"]
    
    # Group by stratification key
    groups = df.groupby("strat_group")
    
    # Initialize folds
    folds = [[] for _ in range(k)]
    fold_counts = {i: defaultdict(int) for i in range(k)}
    
    # For each group, distribute samples into folds
    for group_name, group_df in groups:
        # Shuffle within group
        indices = group_df.index.tolist()
        np.random.shuffle(indices)
        
        # Distribute to folds (round-robin with smallest-first)
        for idx in indices:
            filename = df.loc[idx, "filename"]
            
            # Find fold with fewest samples
            fold_sizes = [len(folds[i]) for i in range(k)]
            min_fold = np.argmin(fold_sizes)
            
            folds[min_fold].append(filename)
            fold_counts[min_fold][group_name] += 1
    
    return folds, fold_counts

print(f"Creating {K_FOLDS} stratified folds (seed={RANDOM_SEED})...")
folds, fold_counts = stratified_kfold_split(df, K_FOLDS, RANDOM_SEED)
print(" K-folds created!")

In [ ]:
# Check balance
print("=" * 80)
print("FOLD BALANCE CHECK")
print("=" * 80)

# Dimensioni fold
print("\nFold Sizes:")
for i, fold in enumerate(folds):
    print(f"   Fold {i:02d}: {len(fold)} samples")

# Distribution table
all_groups = sorted(df["combined_category"].unique())

print(f"\n{'Group':<30} ", end="")
for i in range(K_FOLDS):
    print(f"F{i:02d} ", end="")
print("  Tot")
print("-" * (35 + 4 * K_FOLDS + 7))

group_totals = defaultdict(int)
for group in all_groups:
    print(f"{group:<30} ", end="")
    for i in range(K_FOLDS):
        count = fold_counts[i].get(group, 0)
        group_totals[group] += count
        print(f"{count:3d} ", end="")
    print(f"  {group_totals[group]:4d}")

print("-" * (35 + 4 * K_FOLDS + 7))
print(f"{'TOTAL':<30} ", end="")
for i in range(K_FOLDS):
    print(f"{len(folds[i]):3d} ", end="")
print(f"  {len(df):4d}")

## Phase 5: Saving Results

In [ ]:
# Directory for specific k-folds
kfold_dir = OUTPUT_DIR / f"k{K_FOLDS}_seed{RANDOM_SEED}"
kfold_dir.mkdir(exist_ok=True, parents=True)

print("=" * 60)
print("SAVING RESULTS")
print("=" * 60)

# Save each fold as txt file
print("\nSaving individual folds...")
for i, fold in enumerate(folds):
    fold_file = kfold_dir / f"fold_{i:02d}.txt"
    with open(fold_file, "w") as f:
        for filename in sorted(fold):
            f.write(f"{filename}\n")
    print(f"   - {fold_file.name}")

# Save complete info as JSON
split_info = {
    "num_folds": K_FOLDS,
    "seed": RANDOM_SEED,
    "total_samples": len(df),
    "folds": {f"fold_{i:02d}": sorted(fold) for i, fold in enumerate(folds)}
}

json_file = kfold_dir / "fold_splits.json"
with open(json_file, "w") as f:
    json.dump(split_info, f, indent=2)
print(f"\n {json_file.name}")

In [ ]:
# Generate train/val/test configurations for each fold
# For each fold i:
#   - test: fold i
#   - val: fold (i+1) % k  
#   - train: remaining

print("\n Generating train/val/test configurations...")

configs = []

for test_fold in range(K_FOLDS):
    val_fold = (test_fold + 1) % K_FOLDS
    train_folds = [i for i in range(K_FOLDS) if i != test_fold and i != val_fold]
    
    config = {
        "config_id": test_fold,
        "test_fold": test_fold,
        "val_fold": val_fold,
        "train_folds": train_folds,
        "test_files": sorted(folds[test_fold]),
        "val_files": sorted(folds[val_fold]),
        "train_files": sorted([f for i in train_folds for f in folds[i]])
    }
    configs.append(config)
    
    # Save individual config
    config_file = kfold_dir / f"config_{test_fold:02d}.json"
    with open(config_file, "w") as f:
        json.dump(config, f, indent=2)

# Save summary
summary = {
    "num_folds": K_FOLDS,
    "num_configs": K_FOLDS,
    "seed": RANDOM_SEED,
    "configs": [
        {
            "config_id": c["config_id"],
            "test_fold": c["test_fold"],
            "val_fold": c["val_fold"],
            "train_folds": c["train_folds"],
            "test_count": len(c["test_files"]),
            "val_count": len(c["val_files"]),
            "train_count": len(c["train_files"])
        }
        for c in configs
    ]
}

summary_file = kfold_dir / "configs_summary.json"
with open(summary_file, "w") as f:
    json.dump(summary, f, indent=2)

print(f"   - {K_FOLDS} configurations generated")
print(f"   - {summary_file.name}")

In [ ]:
# Save complete analysis
print("\nSaving complete analysis...")

# CSV for analysis
df.to_csv(OUTPUT_DIR / "steatosis_analysis.csv", index=False)
print(f"   - steatosis_analysis.csv")

# JSON with all data
analysis_data = {
    "thresholds": thresholds,
    "statistics": {
        "total_masks": len(df),
        "black_masks": int((df["num_steatosis"] == 0).sum()),
        "mean_count": float(non_black["num_steatosis"].mean()),
        "mean_area": float(non_black["white_percentage"].mean()),
        "mean_avg_size": float(non_black["avg_steatosis_size"].mean()),
    },
    "category_distribution": {
        "by_count": df["count_category"].value_counts().to_dict(),
        "by_area": df["area_category"].value_counts().to_dict(),
        "combined": df["combined_category"].value_counts().to_dict()
    }
}

with open(OUTPUT_DIR / "steatosis_analysis.json", "w") as f:
    json.dump(analysis_data, f, indent=2)
print(f"   - steatosis_analysis.json")

## Phase 6: Physical File Distribution

Physically copies images and masks into fold-specific directories.

**WARNING**: This operation duplicates data on disk.

In [ ]:
# Physical file distribution
import shutil

print("=" * 60)
print("PHASE 6: PHYSICAL FILE DISTRIBUTION")
print("=" * 60)

for fold_idx, fold in enumerate(folds):
    # Create directories
    fold_dir = kfold_dir / f"fold_{fold_idx:02d}"
    images_subdir = fold_dir / "images"
    masks_subdir = fold_dir / "masks"
    
    images_subdir.mkdir(exist_ok=True, parents=True)
    masks_subdir.mkdir(exist_ok=True, parents=True)
    
    print(f"\nPopulating Fold {fold_idx}...")
    
    count_ok = 0
    for filename in fold:
        # Source paths
        src_image = IMAGES_DIR / filename
        src_mask = MANUAL_DIR / filename
        
        # Destination paths
        dst_image = images_subdir / filename
        dst_mask = masks_subdir / filename
        
        # Copy files
        if src_image.exists() and src_mask.exists():
            shutil.copy2(src_image, dst_image)
            shutil.copy2(src_mask, dst_mask)
            count_ok += 1
        else:
            print(f"   WARNING: Missing file: {filename}")
            
    print(f"   Copied {count_ok} of {len(fold)} images and masks")

print("\nPhysical distribution completed!")

## Final Summary

In [ ]:
print("\n" + "=" * 70)
print("K-FOLD GENERATION COMPLETED!")
print("=" * 70)

print(f"\nSummary:")
print(f"   Masks analyzed: {len(df)}")
print(f"   Number of folds: {K_FOLDS}")
print(f"   Samples per fold: ~{len(df)//K_FOLDS}")
print(f"   Stratified categories: {len(df['combined_category'].unique())}")

print(f"\nOutput Directory: {OUTPUT_DIR}")
print(f"\nGenerated Files:")
print(f"   Analysis:")
print(f"      - steatosis_analysis.csv")
print(f"      - steatosis_analysis.json")
print(f"      - distribution_histograms.png")
print(f"      - category_heatmap.png")
print(f"\n   K-Fold ({kfold_dir.name}):")
print(f"      - fold_XX.txt (filename list per fold)")
print(f"      - fold_splits.json (all folds)")
print(f"      - config_XX.json (train/val/test for each fold)")
print(f"      - configs_summary.json (configuration summary)")

print("\n" + "=" * 70)
print("Ready for balanced K-fold training!")
print("=" * 70)

## Utility: Loading Configurations

Example of how to use generated configurations for training.

In [ ]:
# Example: load configuration for fold 0 as test
def load_fold_config(kfold_dir: Path, config_id: int) -> dict:
    """
    Loads a k-fold configuration.
    
    Args:
        kfold_dir: Directory with configurations
        config_id: Configuration ID (0 to K-1)
        
    Returns:
        Dictionary with train_files, val_files, test_files
    """
    config_file = kfold_dir / f"config_{config_id:02d}.json"
    with open(config_file, "r") as f:
        return json.load(f)

# Load example
example_config = load_fold_config(kfold_dir, 0)

print("Example Configuration 0:")
print(f"   Test fold: {example_config['test_fold']} ({len(example_config['test_files'])} file)")
print(f"   Val fold: {example_config['val_fold']} ({len(example_config['val_files'])} file)")
print(f"   Train folds: {example_config['train_folds']} ({len(example_config['train_files'])} file)")
print(f"\n   First 5 test files: {example_config['test_files'][:5]}")